**Problem Statement:**

Households struggle to sort waste correctly for recycling (e.g., plastic vs. compostable), leading to contamination and reduced recycling rates, especially in regions with complex waste regulations.

**Solution:**
Develop a computer vision system using a supervised deep learning model (e.g., convolutional neural network) to classify waste items from smartphone camera images. Train on a custom dataset of labeled waste images (e.g., plastic, organic, metal) to identify materials and provide real-time sorting instructions via a mobile app. Integrate with local recycling rules to offer tailored advice (e.g., “Rinse this plastic bottle before recycling”). Use simulation to model waste sorting accuracy and optimize user feedback.

**Impact for Humans:**
Simplifies recycling, reduces landfill waste, and promotes environmental awareness, especially in urban households.

In [ ]:
#!pip install numpy
#!pip install pandas
#!pip install tensorflow
#!pip install opencv-python
#!pip install albumentations
#!pip install scikit-learn

In [2]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import pandas as pd
import cv2
import os
from sklearn.metrics import classification_report, confusion_matrix

# 1. Set up paths and parameters
DATA_DIR = '/content/drive/MyDrive/Projects/Smart Waste Sorting/Dataset/subset_data/augmented_data/'  # Folder with subfolders: plastic/, organic/, etc.
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

# 2. Load and preprocess data
def load_data():
    datagen = ImageDataGenerator(
        rescale=1./255,
        validation_split=0.2,
        rotation_range=20,
        zoom_range=0.2,
        horizontal_flip=True
    )

    train_generator = datagen.flow_from_directory(
        DATA_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training'
    )

    val_generator = datagen.flow_from_directory(
        DATA_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation'
    )

    return train_generator, val_generator

# 3. Build the model
def build_model():
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False  # Freeze base model

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    predictions = Dense(len(CLASSES), activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# 4. Train the model
def train_model(model, train_generator, val_generator):
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        'waste_classifier.h5', save_best_only=True, monitor='val_accuracy'
    )
    history = model.fit(
        train_generator,
        epochs=EPOCHS,
        validation_data=val_generator,
        callbacks=[checkpoint]
    )
    return history

# 5. Evaluate the model
def evaluate_model(model, test_dir):
    test_datagen = ImageDataGenerator(rescale=1./255)
    test_generator = test_datagen.flow_from_directory(
        test_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )

    predictions = model.predict(test_generator)
    y_pred = np.argmax(predictions, axis=1)
    y_true = test_generator.classes
    print(classification_report(y_true, y_pred, target_names=CLASSES))
    print(confusion_matrix(y_true, y_pred))

# 6. Convert to TensorFlow Lite
def convert_to_tflite(model_path, output_path='waste_classifier.tflite'):
    model = tf.keras.models.load_model(model_path)
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    with open(output_path, 'wb') as f:
        f.write(tflite_model)

# 7. Test the TFLite model
def test_tflite_model(tflite_path, test_image_path):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    img = cv2.imread(test_image_path)
    img = cv2.resize(img, IMG_SIZE)
    img = img / 255.0
    img = img.astype(np.float32).reshape(1, 224, 224, 3)

    interpreter.set_tensor(input_details[0]['index'], img)
    interpreter.invoke()
    prediction = interpreter.get_tensor(output_details[0]['index'])
    category = CLASSES[np.argmax(prediction)]
    return category

# Main execution
if __name__ == "__main__":
    # Load data
    train_generator, val_generator = load_data()

    # Build and train model
    model = build_model()
    history = train_model(model, train_generator, val_generator)

Found 4800 images belonging to 6 classes.
Found 1200 images belonging to 6 classes.
Epoch 1/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.5924 - loss: 1.0864

150/150 ━━━━━━━━━━━━━━━━━━━━ 1247s 8s/step - accuracy: 0.5931 - loss: 1.0846 - val_accuracy: 0.6758 - val_loss: 0.9147
Epoch 2/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8560 - loss: 0.4217

150/150 ━━━━━━━━━━━━━━━━━━━━ 336s 2s/step - accuracy: 0.8561 - loss: 0.4216 - val_accuracy: 0.7125 - val_loss: 0.8675
Epoch 3/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8992 - loss: 0.3112

150/150 ━━━━━━━━━━━━━━━━━━━━ 333s 2s/step - accuracy: 0.8992 - loss: 0.3111 - val_accuracy: 0.7283 - val_loss: 0.8635
Epoch 4/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 332s 2s/step - accuracy: 0.9293 - loss: 0.2212 - val_accuracy: 0.7217 - val_loss: 0.9195
Epoch 5/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 336s 2s/step - accuracy: 0.9364 - loss: 0.1979 - val_accuracy: 0.7108 - val_loss: 1.0215
Epoch 6/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 383s 2s/step - accuracy: 0.9433 - loss: 0.1676 - val_accuracy: 0.7150 - val_loss: 1.0314
Epoch 7/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 334s 2s/step - accuracy: 0.9551 - loss: 0.1279 - val_accuracy: 0.7158 - val_loss: 0.9962
Epoch 8/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 335s 2s/step - accuracy: 0.9593 - loss: 0.1357 - val_accuracy: 0.7117 - val_loss: 1.0478
Epoch 9/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 341s 2s/step - accuracy: 0.9578 - loss: 0.1213 - val_accuracy: 0.7033 - val_loss: 1.1018
Epoch 10/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 339s 2s/step - accuracy: 0.9640 - loss: 0.1115 - val_accuracy: 0.70

150/150 ━━━━━━━━━━━━━━━━━━━━ 331s 2s/step - accuracy: 0.9735 - loss: 0.0742 - val_accuracy: 0.7325 - val_loss: 1.1295


In [4]:
# Evaluate on test set
evaluate_model(model, '/content/drive/MyDrive/Projects/Smart Waste Sorting/Dataset/subset_data/Test/')

# Convert to TFLite
convert_to_tflite('waste_classifier.h5')

# Test TFLite model
test_image = '/content/drive/MyDrive/Projects/Smart Waste Sorting/Dataset/subset_data/Test/plastic/plastic30.jpg'
predicted_category = test_tflite_model('waste_classifier.tflite', test_image)
print(f"Predicted category: {predicted_category}, Actual Category: plastic")

Found 600 images belonging to 6 classes.
19/19 ━━━━━━━━━━━━━━━━━━━━ 97s 5s/step
              precision    recall  f1-score   support

   cardboard       0.87      0.97      0.92       100
       glass       0.84      0.81      0.82       100
       metal       0.80      0.82      0.81       100
       paper       0.84      0.82      0.83       100
     plastic       0.83      0.78      0.80       100
       trash       0.89      0.87      0.88       100

    accuracy                           0.84       600
   macro avg       0.84      0.85      0.84       600
weighted avg       0.84      0.84      0.84       600

[[97  0  0  3  0  0]
 [ 0 81  7  0 10  2]
 [ 1  7 82  4  4  2]
 [10  0  3 82  0  5]
 [ 1  9  4  6 78  2]
 [ 2  0  6  3  2 87]]


Saved artifact at '/tmp/tmp_mfx15yj'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  135669967404880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135669967402000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135669967401040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135669967403344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135669967393744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135669967399504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135669967392976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135669967392400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135669967391248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135669967390096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1356699674044